In [2]:
import re
import pandas as pd
import dotenv
import os
from neo4j import GraphDatabase

In [7]:
triplets_df_path = "../data/TRIPLETS_ALL.csv"
env_file = "../Neo4j_private.txt"

In [5]:
df = pd.read_csv(triplets_df_path)

# clear neo4j 

In [6]:
dotenv.load_dotenv(env_file)

True

In [7]:
from neo4j import GraphDatabase

dotenv.load_dotenv(env_file)
URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(URI, auth=(USER, PWD))

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

driver.close()

# ------

In [18]:
df = pd.read_csv(triplets_df_path)

In [9]:
#df.loc[
#    (df['entity1_type'] == 'country')]["entity1"].unique()

In [19]:
def to_camel_case(s: str) -> str:
    return ''.join(word.capitalize() for word in s.split('_'))


In [20]:
df['entity2_type'] = df['entity2_type'].apply(to_camel_case)
df['entity1_type'] = df['entity1_type'].apply(to_camel_case)


In [21]:
df.loc[
    (df['entity1_type'] == 'Country') |
    (df['entity2_type'] == 'Country')
] 

,entity1,entity1_type,rel_type,entity2,entity2_type,sector,url,date
14,First Iraqi Bank,Company,is_member_of,Iraq,Country,financials,https://gfmag.com/executive-interviews/complia...,2025-12-30T15:24:25+00:00
16,First Iraqi Bank,Company,has_positive_impact,Iraq,Country,financials,https://gfmag.com/executive-interviews/complia...,2025-12-30T15:24:25+00:00
18,First Iraqi Bank,Company,has_positive_impact,Iraq,Country,financials,https://gfmag.com/executive-interviews/complia...,2025-12-30T15:24:25+00:00
20,First Iraqi Bank,Company,has_positive_impact,Iraq,Country,financials,https://gfmag.com/executive-interviews/complia...,2025-12-30T15:24:25+00:00
28,Financial Action Task Force,Regulator,controls,South Africa,Country,financials,https://gfmag.com/news/fatf-removes-4-countrie...,2025-12-19T17:47:20+00:00
...,...,...,...,...,...,...,...,...
30783,Singapore,Country,is_member_of,Asean economic region,EconomicIndicator,public_sector,https://gfmag.com/data/shifting-fortunes/,2017-07-21T00:00:00+00:00
30784,Singapore,Country,is_member_of,Association of Southeast Asian Nations,EconomicIndicator,public_sector,https://gfmag.com/data/shifting-fortunes/,2017-07-21T00:00:00+00:00
30785,Singapore,Country,is_member_of,Asean economic region,EconomicIndicator,public_sector,https://gfmag.com/data/shifting-fortunes/,2017-07-21T00:00:00+00:00
30786,Singapore,Country,is_member_of,Association of Southeast Asian Nations,EconomicIndicator,public_sector,https://gfmag.com/data/shifting-fortunes/,2017-07-21T00:00:00+00:00


In [83]:
import pycountry

In [56]:
pycountry.countries.lookup("AI").name

'Anguilla'

In [73]:
def normalize_countries(name: str, entity_type: str):
    """
    Conservative country normalization.
    Never upgrades non-country entities.
    """
    cleaned = re.sub(r"[^\w\s]", "", name)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    if entity_type != "Country":
        return cleaned, entity_type
    try:
        country = pycountry.countries.lookup(cleaned)
        return country.name, "Country"
    except:
        return cleaned, "Other"

In [72]:
e1_name, e1_type

('Nicaragua', 'country')

In [63]:
country = pycountry.countries.lookup("AI")

In [64]:
country

Country(alpha_2='AI', alpha_3='AIA', flag='🇦🇮', name='Anguilla', numeric='660')

In [65]:
country.name

'Anguilla'

In [16]:
df.dropna(subset=["entity1", "entity2"], inplace=True)

In [87]:
df[["entity1", "entity1_type"]] = df.apply(
    lambda row: normalize_countries(row["entity1"], row["entity1_type"]),
    axis=1,
    result_type="expand"
)
df[["entity2", "entity2_type"]] = df.apply(
    lambda row: normalize_countries(row["entity2"], row["entity2_type"]),
    axis=1,
    result_type="expand"
)

In [22]:
df.to_csv("../data/TRIPLETS_ALL.csv", index=False)

In [95]:
df.loc[df["rel_type"]=="invests_in"]

,entity1,entity1_type,rel_type,entity2,entity2_type,sector,url,date
10,corporates,company,invests_in,digital tools,product_service,technology,https://gfmag.com/executive-interviews/standar...,2025-12-31T18:00:00+00:00
22,Mirae Asset Securities,company,invests_in,artificial intelligence,product_service,technology,https://gfmag.com/banking/mirae-asset-securiti...,2025-12-16T11:02:48+00:00
58,Qatar Investment Authority,financial_institution,invests_in,Anthropic,company,technology,https://gfmag.com/features/qatar-new-world-cap...,2025-12-10T20:41:50+00:00
59,Qatar Investment Authority,financial_institution,invests_in,Applied Intuition,company,technology,https://gfmag.com/features/qatar-new-world-cap...,2025-12-10T20:41:50+00:00
181,Banks,financial_institution,invests_in,Private credit,financial_instrument,financials,https://gfmag.com/banking/familiar-bedfellows/,2025-10-08T15:04:07+00:00
...,...,...,...,...,...,...,...,...
30624,FAB,financial_institution,invests_in,cash management and liquidity solutions,product_service,financials,https://gfmag.com/award/best-treasury-cash-man...,2024-07-26T15:37:10+00:00
30720,Citi,financial_institution,invests_in,cloudbased flexible microservices,product_service,technology,https://gfmag.com/executive-interviews/reaping...,2023-09-03T00:00:00+00:00
30772,Nordea Bank,financial_institution,invests_in,blockchain,financial_instrument,technology,https://gfmag.com/executive-interviews/nordea-...,2019-12-09T00:00:00+00:00
30790,Data Collective venture capital fund,financial_institution,invests_in,Merlon Intelligence,company,financials,https://gfmag.com/data/can-technology-ensure-c...,2017-07-21T00:00:00+00:00


In [15]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

BASE_URL = "https://finance.yahoo.com/news/"
DELAY_SEC = 1.0  # polite delay between scrolls

def collect_yahoo_news_selenium(max_articles=5000, out_csv="yahoo_news_selenium.csv"):
    # Configure Chrome for headless mode
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")  # runs Chrome in headless mode
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=chrome_options)
    driver.get(BASE_URL)
    time.sleep(3)  # wait for initial content to load

    seen = set()
    rows = []

    last_height = driver.execute_script("return document.body.scrollHeight")
    scroll_attempts = 0

    while len(seen) < max_articles and scroll_attempts < 200:
        # collect links
        anchors = driver.find_elements(By.XPATH, '//a[contains(@href, "/news/")]')
        for a in anchors:
            url = a.get_attribute("href")
            if url not in seen:
                seen.add(url)
                rows.append({"Url": url})
                print(f"Collected {len(seen)} articles", end="\r")

        # scroll down
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(DELAY_SEC)

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            scroll_attempts += 1  # stop after a few scrolls without new content
        else:
            scroll_attempts = 0
            last_height = new_height

    driver.quit()

    df = pd.DataFrame(rows[:max_articles])
    df["Text"] = ""
    df["Date"] = ""
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {len(df)} articles to {out_csv}")


if __name__ == "__main__":
    collect_yahoo_news_selenium(max_articles=500, out_csv="yahoo_news_test.csv")


Defaulting to user installation because normal site-packages is not writeable


In [34]:
import re
import csv
from time import sleep
from bs4 import BeautifulSoup
import requests

In [37]:
import requests

url = "https://finance.yahoo.com/news/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("Status:", response.status_code)
print(response.text[:500])  # print the first 500 characters for inspection


Status: 200
<!doctype html>
<html lang="en-US" theme="auto" data-color-theme-enabled="true" data-color-scheme="auto" class="desktop neo-green dock-upscale">
    <head>
        <meta charset="utf-8" />
        <meta name="oath:guce:consent-host" content="guce.yahoo.com" />
        <link rel="preconnect" href="//s.yimg.com" crossorigin="anonymous"><link rel="preconnect" href="//geo.yahoo.com"/><link rel="preconnect" href="//query1.finance.yahoo.com"/><link rel="preconnect" href="//consent.cmp.oath.com"/><link


In [38]:
html = requests.get(url, headers=headers).text
soup = BeautifulSoup(html, "html.parser")

lis = soup.find_all("li")
print("Total <li> tags:", len(lis))

Total <li> tags: 815


In [54]:
tickers_df = pd.read_csv("tickerssp500.csv")
tickers = list(tickers_df["Symbol"].str.lower())

In [58]:
import requests
from bs4 import BeautifulSoup

rss_base = "https://feeds.finance.yahoo.com/rss/2.0/headline"

headers = {"User-Agent": "Mozilla/5.0"}
all_links = set()

for ticker in tickers:
    url = f"{rss_base}?s={ticker}&region=US&lang=en-US"
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.content, "xml")
    for item in soup.find_all("item"):
        all_links.add(item.link.text)


In [60]:
df = pd.DataFrame(list(all_links), columns=["link"])
df.to_csv("yahoo_links.csv", index=False)

In [61]:
import requests
from bs4 import BeautifulSoup

url = "https://finance.yahoo.com/news/3m-company-stock-wall-street-150707139.html"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/144.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("Status:", response.status_code)

html = response.text
soup = BeautifulSoup(html, "html.parser")


Status: 200


In [65]:
# Find all paragraph tags
paragraphs = soup.find_all("p")

# Join text
article_text = "\n".join(p.get_text(strip=True) for p in paragraphs)

print(article_text[:2000])  # preview first 1000 chars

Oops, something went wrong
Saint Paul, Minnesota-based 3M Company (MMM) is a diversified technology company with a market cap of $81.4 billion. It develops a wide range of products serving industrial, safety, consumer, and electronics markets.
This industrial company has underperformed the broader market over the past 52 weeks. Shares of MMM have gained marginally over this time frame, while the broader S&P 500 Index ($SPX) has surged 14.3%. Moreover, on a YTD basis, the stock is down 4.4%, compared to SPX’s 1.4% uptick.
Chevron Hikes Its Dividend - But It's Less Than Expected - Is CVX Stock Fully Valued?
Before Amazon Invests $50 Billion in OpenAI, How Should You Play AMZN Stock?
Commodity Volatility, Earnings and Other Key Things to Watch
Stop Missing Market Moves: Get the FREE Barchart Brief – your midday dose of stock movers, trending sectors, and actionable trade ideas, delivered right to your inbox. Sign Up Now!
Narrowing the focus, MMM has also lagged behind the State Street Ind